# Notebook 05 — CZ-Like Gate Benchmark

This notebook extends the Rydberg model to a simple **gate-level benchmark**.

Rather than only tracking blockade or Bell-state fidelity, we now compare the evolution of the computational basis states:

$$
|gg\rangle,\ |gr\rangle,\ |rg\rangle,\ |rr\rangle
$$

and evaluate how well the dynamics approximates a **CZ-like entangling operation**.

This notebook is not a full pulse-engineered gate design. Instead, it is a compact benchmark that asks:

- how distinguishable are the four basis-state responses,
- how does the interaction $V$ affect the two-body phase structure,
- how do decay and dephasing reduce gate quality?

We use a phase-sensitive overlap metric as a simple proxy for gate performance.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'scipy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qutip import basis, qeye, tensor, sigmax, sesolve, mesolve, fidelity

## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()
sigma_minus = g * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = {
    '|gg>': gg,
    '|gr>': gr,
    '|rg>': rg,
    '|rr>': rr,
}

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)


## Hamiltonian and collapse operators

In [ ]:
def two_atom_hamiltonian(omega: float, delta: float, V: float):
    drive = 0.5 * omega * (sx1 + sx2)
    detuning = -delta * (n1 + n2)
    interaction = V * n1 * n2
    return drive + detuning + interaction


def collapse_operators(gamma_decay: float = 0.0, gamma_phi: float = 0.0):
    c_ops = []
    if gamma_decay > 0:
        c_ops.append(np.sqrt(gamma_decay) * sm1)
        c_ops.append(np.sqrt(gamma_decay) * sm2)
    if gamma_phi > 0:
        c_ops.append(np.sqrt(gamma_phi) * n1)
        c_ops.append(np.sqrt(gamma_phi) * n2)
    return c_ops

## CZ-like benchmark metric

For an ideal CZ gate, the computational basis populations are preserved, while the $|11\rangle$ state acquires a minus sign.

In this toy encoding, we use $|g\rangle \leftrightarrow |0\rangle$ and $|r\rangle \leftrightarrow |1\rangle$.

We define a simple **state-by-state benchmark**:

- evolve each basis input state,
- compare the output against the corresponding ideal CZ target,
- average the four state fidelities.

This is a useful first proxy for gate quality, even though it is not a full process tomography calculation.


In [ ]:
ideal_cz_targets = {
    '|gg>': gg,
    '|gr>': gr,
    '|rg>': rg,
    '|rr>': -rr,
}


def evolve_state(psi0, omega: float, delta: float, V: float,
                 gamma_decay: float = 0.0,
                 gamma_phi: float = 0.0,
                 t_final: float = 4.0,
                 n_steps: int = 200):
    H = two_atom_hamiltonian(omega, delta, V)
    c_ops = collapse_operators(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    times = np.linspace(0.0, t_final, n_steps)

    if len(c_ops) == 0:
        result = sesolve(H, psi0, times)
        final_state = result.states[-1]
    else:
        result = mesolve(H, psi0, times, c_ops)
        final_state = result.states[-1]

    return times, result.states, final_state


def state_fidelity_to_target(final_state, target_state):
    return float(np.real(fidelity(final_state, target_state)) ** 2)


def cz_like_score(omega: float, delta: float, V: float,
                  gamma_decay: float = 0.0,
                  gamma_phi: float = 0.0,
                  t_final: float = 4.0,
                  n_steps: int = 200):
    scores = {}
    for label, psi0 in basis_states.items():
        _, _, final_state = evolve_state(
            psi0,
            omega=omega,
            delta=delta,
            V=V,
            gamma_decay=gamma_decay,
            gamma_phi=gamma_phi,
            t_final=t_final,
            n_steps=n_steps,
        )
        scores[label] = state_fidelity_to_target(final_state, ideal_cz_targets[label])

    avg_score = float(np.mean(list(scores.values())))
    return avg_score, scores

## Basis-state benchmark at one operating point

In [ ]:
omega = 1.0
delta = 0.0
V = 4.0
t_final = 4.0

avg_score, scores = cz_like_score(omega=omega, delta=delta, V=V, t_final=t_final)
print(f'Average CZ-like score: {avg_score:.4f}')
for label, score in scores.items():
    print(f'{label}: {score:.4f}')

## Final-state fidelities by basis input

In [ ]:
labels = list(scores.keys())
values = [scores[k] for k in labels]

plt.figure(figsize=(7, 4.5))
plt.bar(labels, values)
plt.ylim(0, 1.05)
plt.ylabel('State fidelity to ideal CZ target')
plt.title('Basis-state benchmark at a fixed operating point')
plt.tight_layout()
plt.show()

## Average CZ-like score over $(\Omega, V)$

In [ ]:
omega_vals = np.linspace(0.2, 2.5, 32)
V_vals = np.linspace(0.0, 8.0, 32)

cz_landscape = np.zeros((len(V_vals), len(omega_vals)))

for i, V in enumerate(V_vals):
    for j, omega in enumerate(omega_vals):
        avg_score, _ = cz_like_score(
            omega=omega,
            delta=0.0,
            V=V,
            gamma_decay=0.0,
            gamma_phi=0.0,
            t_final=4.0,
            n_steps=160,
        )
        cz_landscape[i, j] = avg_score

In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(
    cz_landscape,
    origin='lower',
    aspect='auto',
    extent=[omega_vals[0], omega_vals[-1], V_vals[0], V_vals[-1]],
)
plt.contour(omega_vals, V_vals, cz_landscape, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Average CZ-like score over $(\Omega, V)$')
plt.colorbar(im, label='average score')
plt.tight_layout()
plt.show()

## Noise sensitivity of the CZ-like score

In [ ]:
gamma_decay_grid = np.linspace(0.0, 0.15, 26)
gamma_phi_grid = np.linspace(0.0, 0.15, 26)

noise_landscape = np.zeros((len(gamma_phi_grid), len(gamma_decay_grid)))

for i, gamma_phi in enumerate(gamma_phi_grid):
    for j, gamma_decay in enumerate(gamma_decay_grid):
        avg_score, _ = cz_like_score(
            omega=1.0,
            delta=0.0,
            V=4.0,
            gamma_decay=gamma_decay,
            gamma_phi=gamma_phi,
            t_final=4.0,
            n_steps=160,
        )
        noise_landscape[i, j] = avg_score

In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(
    noise_landscape,
    origin='lower',
    aspect='auto',
    extent=[gamma_decay_grid[0], gamma_decay_grid[-1], gamma_phi_grid[0], gamma_phi_grid[-1]],
)
plt.contour(gamma_decay_grid, gamma_phi_grid, noise_landscape, colors='white', linewidths=0.4)
plt.xlabel(r'Spontaneous emission $\gamma$')
plt.ylabel(r'Pure dephasing $\gamma_\phi$')
plt.title(r'Average CZ-like score over $(\gamma, \gamma_\phi)$')
plt.colorbar(im, label='average score')
plt.tight_layout()
plt.show()

## Scan gate time at one operating point

In [ ]:
gate_times = np.linspace(0.5, 6.0, 45)
scores_vs_time = []

for t_gate in gate_times:
    avg_score, _ = cz_like_score(
        omega=1.0,
        delta=0.0,
        V=4.0,
        gamma_decay=0.0,
        gamma_phi=0.0,
        t_final=t_gate,
        n_steps=180,
    )
    scores_vs_time.append(avg_score)

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(gate_times, scores_vs_time)
plt.xlabel('Gate time')
plt.ylabel('Average CZ-like score')
plt.title('Gate-time dependence at a fixed operating point')
plt.tight_layout()
plt.show()

## Best point in the coherent $(\Omega, V)$ scan

In [ ]:
best_index = np.unravel_index(np.argmax(cz_landscape), cz_landscape.shape)
best_V = V_vals[best_index[0]]
best_omega = omega_vals[best_index[1]]
best_score = cz_landscape[best_index]

print(f'Best coherent scan point: Omega = {best_omega:.3f}, V = {best_V:.3f}, average CZ-like score = {best_score:.4f}')

## Interpretation

This notebook adds a gate-oriented view of the same physical model:

- basis-state fidelities show whether the dynamics approximately preserves the computational basis structure,
- the $|rr\rangle$ channel is the most sensitive to the interaction-induced phase,
- the $(\Omega, V)$ landscape identifies regions where a CZ-like benchmark looks strongest,
- the noise landscape shows how decay and dephasing reduce average gate quality.

This provides a simple bridge from Rydberg blockade physics to entangling-gate characterization.


## Optional next steps

Natural follow-ups are:

- replace this CZ-like proxy with a full process-fidelity calculation,
- add pulse shaping $\Omega(t)$ and detuning ramps $\Delta(t)$,
- include distance-dependent interactions $V(R)$,
- compare this toy benchmark against a more realistic neutral-atom gate sequence.
